# 08. Single-Molecule Age Prediction

Score individual long reads using the compact age reference and visualize per-read methylation patterns at clock loci.

**Key ideas:**
- Each read covering >= 5 eligible CpGs is assigned a maximum-likelihood age estimate
- Fisher information selects the most age-informative CpG windows
- ELOVL2 read-level heatmaps reveal the bimodal C/T signature of aging

**Inputs:**
- Compact references from `results/age_references_v2/` (notebook 05)
- Holdout test set beta + PAT files

**Outputs:**
- Per-read age estimates CSV
- ELOVL2 heatmaps (PDF)

---

### Config, imports, and reset

In [ ]:
# ============================================================
# 0) FULL RESET: CONFIG + IMPORTS
# ============================================================
import os
import io
import json
import gzip
import glob
import math
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from google.cloud import storage

warnings.filterwarnings("ignore")

# -----------------------------
# Required AoU environment
# -----------------------------
WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
assert WORKSPACE_BUCKET.startswith("gs://"), WORKSPACE_BUCKET

client = storage.Client()

# -----------------------------
# Reference config
# -----------------------------
TARGET_TECH = "ONT"
OUTPUT_ROOT = "results/age_references_v2"
REFERENCE_INDEX_GS = f"{WORKSPACE_BUCKET}/{OUTPUT_ROOT}/reference_index.csv"

# -----------------------------
# Resource restore config
# -----------------------------
LOCAL_RESOURCE_DIR = "/home/jupyter/wgbs_resources"

# -----------------------------
# Window selection config
# -----------------------------
WINDOW_BP = 10_000
STEP_BP = 2_500

FI_AGE_MIN = 20
FI_AGE_MAX = 100
FI_AGE_STEP = 1.0

# -----------------------------
# Read-level prediction config
# -----------------------------
MIN_CPGS_PER_READ = 3
K_INFER = 500

# Start small. Set to None after this works.
N_TEST_SAMPLES = 20

# -----------------------------
# ELOVL2 diagnostic
# -----------------------------
ELOVL2_SITE_HG38 = 11044644
ELOVL2_WINDOW_BP = 5000

BOUNDARY_Q_LOW = 0.10
BOUNDARY_Q_HIGH = 0.90

# -----------------------------
# Output
# -----------------------------
OUT_DIR = "/home/jupyter/single_molecule_region_diagnostics_global_index"
os.makedirs(OUT_DIR, exist_ok=True)

print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET)
print("REFERENCE_INDEX_GS:", REFERENCE_INDEX_GS)
print("LOCAL_RESOURCE_DIR:", LOCAL_RESOURCE_DIR)
print("OUT_DIR:", OUT_DIR)
print("MIN_CPGS_PER_READ:", MIN_CPGS_PER_READ)

### GCS helpers

In [ ]:
# ============================================================
# 1) GCS HELPERS
# ============================================================
def gs_to_bucket_blob(gs_path: str):
    assert gs_path.startswith("gs://"), gs_path
    rest = gs_path[5:]
    bkt, obj = rest.split("/", 1)
    return bkt, obj

def gcs_blob_exists(gs_path: str) -> bool:
    bkt, obj = gs_to_bucket_blob(gs_path)
    return client.bucket(bkt).blob(obj).exists()

def download_text_from_gcs(gs_path: str):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    if not blob.exists():
        return None
    return blob.download_as_text()

def download_bytes_from_gcs(gs_path: str) -> bytes:
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    if not blob.exists():
        raise FileNotFoundError(gs_path)
    return blob.download_as_bytes()

def read_csv_from_gcs_text(gs_path: str) -> pd.DataFrame:
    txt = download_text_from_gcs(gs_path)
    if txt is None:
        raise FileNotFoundError(gs_path)
    return pd.read_csv(io.StringIO(txt))

def load_json_from_gcs(gs_path: str):
    txt = download_text_from_gcs(gs_path)
    if txt is None:
        raise FileNotFoundError(gs_path)
    return json.loads(txt)

def load_npy_from_gcs(gs_path: str):
    data = download_bytes_from_gcs(gs_path)
    return np.load(io.BytesIO(data), allow_pickle=False)

def resolve_pat_path_from_beta(beta_gs: str):
    if not isinstance(beta_gs, str) or not beta_gs.endswith(".beta"):
        return None

    cand1 = beta_gs[:-5] + ".pat.gz"
    cand2 = beta_gs[:-5] + ".pat"

    if gcs_blob_exists(cand1):
        return cand1
    if gcs_blob_exists(cand2):
        return cand2
    return None

### Restore wgbstools + hg38 bundle from workspace bucket

In [ ]:
# ============================================================
# 2) RESTORE WGBSTOOLS + hg38 BUNDLE FROM WORKSPACE BUCKET
# ============================================================
os.makedirs(LOCAL_RESOURCE_DIR, exist_ok=True)

resources = {
    "wgbstools": f"{WORKSPACE_BUCKET}/resources/wgbstools_package.tar.gz",
    "hg38": f"{WORKSPACE_BUCKET}/resources/hg38_bundle.tar.gz",
}

wgb_tar = f"{LOCAL_RESOURCE_DIR}/wgbstools_package.tar.gz"
hg38_tar = f"{LOCAL_RESOURCE_DIR}/hg38_bundle.tar.gz"

print("Resource paths:")
print(resources)

if not os.path.exists(wgb_tar):
    print("Copying wgbstools package...")
    subprocess.run(["gsutil", "cp", resources["wgbstools"], wgb_tar], check=True)
else:
    print("wgbstools tarball already exists:", wgb_tar)

if not os.path.exists(hg38_tar):
    print("Copying hg38 bundle...")
    subprocess.run(["gsutil", "cp", resources["hg38"], hg38_tar], check=True)
else:
    print("hg38 tarball already exists:", hg38_tar)

print("Extracting wgbstools package...")
subprocess.run(["tar", "-xzf", wgb_tar, "-C", LOCAL_RESOURCE_DIR], check=True)

print("Extracting hg38 bundle...")
subprocess.run(["tar", "-xzf", hg38_tar, "-C", LOCAL_RESOURCE_DIR], check=True)

wgb_bins = sorted(glob.glob(f"{LOCAL_RESOURCE_DIR}/**/wgbstools", recursive=True))
wgb_bins = [x for x in wgb_bins if os.path.isfile(x)]

if len(wgb_bins) == 0:
    raise FileNotFoundError("Could not find wgbstools binary after extraction.")

TOOL_BIN = wgb_bins[0]
os.chmod(TOOL_BIN, 0o755)

fasta_candidates = []
fasta_candidates += glob.glob(f"{LOCAL_RESOURCE_DIR}/**/hg38.fasta", recursive=True)
fasta_candidates += glob.glob(f"{LOCAL_RESOURCE_DIR}/**/*.fasta", recursive=True)
fasta_candidates += glob.glob(f"{LOCAL_RESOURCE_DIR}/**/*.fa", recursive=True)
fasta_candidates = sorted(set([x for x in fasta_candidates if os.path.isfile(x)]))

if len(fasta_candidates) == 0:
    raise FileNotFoundError("Could not find hg38 FASTA after extraction.")

GENOME_FILE = fasta_candidates[0]

print("TOOL_BIN:", TOOL_BIN)
print("GENOME_FILE:", GENOME_FILE)

cmd = [TOOL_BIN, "init_genome", "hg38", "--fasta_path", GENOME_FILE, "-f"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Resource restore complete.")

### Load saved bulk ONT reference + test manifest

In [ ]:
# ============================================================
# 4) LOAD SAVED BULK ONT REFERENCE + TEST MANIFEST
# ============================================================
ref_index = read_csv_from_gcs_text(REFERENCE_INDEX_GS)

ont_rows = ref_index[
    (ref_index["modality"] == "bulk") &
    (ref_index["celltype"] == "bulk") &
    (ref_index["group_level"] == "technology") &
    (ref_index["group_name"] == TARGET_TECH)
].copy()

if len(ont_rows) == 0:
    raise ValueError("No saved bulk ONT pooled reference found in reference_index.csv.")

ont_row = ont_rows.iloc[0].copy()
display(pd.DataFrame([ont_row]))

compact_prefix = str(ont_row["compact_prefix"])
if not compact_prefix.endswith("/"):
    compact_prefix += "/"

eligible_idx0 = load_npy_from_gcs(f"{compact_prefix}eligible_idx.npy").astype(np.int64)
intercept_eligible = load_npy_from_gcs(f"{compact_prefix}intercept_eligible.npy").astype(np.float64)
coef_eligible = load_npy_from_gcs(f"{compact_prefix}coef_eligible.npy").astype(np.float64)
t_stat_eligible = load_npy_from_gcs(f"{compact_prefix}t_stat_eligible.npy").astype(np.float64)
age_grid_ref = load_npy_from_gcs(f"{compact_prefix}age_grid.npy").astype(np.float64)

scalars = load_json_from_gcs(f"{compact_prefix}scalars.json")
EPS = float(scalars["EPS"])

test_manifest = read_csv_from_gcs_text(ont_row["test_manifest_gs"])
test_manifest["person_id"] = test_manifest["person_id"].astype(str)
test_manifest["Age"] = pd.to_numeric(test_manifest["Age"], errors="coerce")
test_manifest = test_manifest.dropna(subset=["Age", "bulk_beta_path"]).copy()

if N_TEST_SAMPLES is None:
    test_manifest_run = test_manifest.copy()
else:
    test_manifest_run = test_manifest.head(N_TEST_SAMPLES).copy()

print("Eligible ONT reference CpGs:", len(eligible_idx0))
print("Held-out ONT samples total:", len(test_manifest))
print("Held-out ONT samples used now:", len(test_manifest_run))
print("Age grid range:", age_grid_ref.min(), age_grid_ref.max(), "n =", len(age_grid_ref))
display(test_manifest_run.head())

### Compute per-site Fisher information

In [ ]:
# ============================================================
# 5) COMPUTE PER-SITE FISHER INFORMATION
# ============================================================
age_grid_fi = np.arange(FI_AGE_MIN, FI_AGE_MAX + 1e-12, FI_AGE_STEP, dtype=np.float64)

P = intercept_eligible[:, None] + coef_eligible[:, None] * age_grid_fi[None, :]
P = np.clip(P, EPS, 1.0 - EPS)

I_age = (coef_eligible[:, None] ** 2) / (P * (1.0 - P))
I_avg = I_age.mean(axis=1)

site_fi_df = pd.DataFrame({
    "eligible_pos": np.arange(len(eligible_idx0), dtype=np.int64),
    "eligible_idx0": eligible_idx0,
    "global_idx_1based": eligible_idx0 + 1,
    "intercept": intercept_eligible,
    "coef": coef_eligible,
    "t_stat": t_stat_eligible,
    "I_avg_uniform_20_100": I_avg,
})

eligible_global1_set = set(site_fi_df["global_idx_1based"].astype(int).tolist())

print("site_fi_df:", site_fi_df.shape)
display(site_fi_df.head())

### Map eligible sites to genome using global CpG indices

In [ ]:
# ============================================================
# 6) MAP ELIGIBLE SITES TO GENOME USING GLOBAL CpG INDICES ONLY
#    This is intentionally chunked and global-index based.
# ============================================================
if not os.path.exists(CPG_BED):
    raise FileNotFoundError(CPG_BED)

compression = "gzip" if CPG_BED.endswith(".gz") else None

mapped_chunks = []
chunksize = 2_000_000

reader = pd.read_csv(
    CPG_BED,
    sep="\t",
    header=None,
    compression=compression,
    chunksize=chunksize
)

for chunk in tqdm(reader, desc="Mapping eligible CpGs from CpG.bed"):
    if chunk.shape[1] < 3:
        raise ValueError(f"Unexpected CpG BED chunk shape: {chunk.shape}")

    chunk = chunk.iloc[:, :3].copy()
    chunk.columns = ["chrom", "pos", "global_idx_1based"]

    chunk["global_idx_1based"] = pd.to_numeric(chunk["global_idx_1based"], errors="coerce")
    chunk = chunk.dropna(subset=["global_idx_1based"]).copy()
    chunk["global_idx_1based"] = chunk["global_idx_1based"].astype(np.int64)

    keep = chunk["global_idx_1based"].isin(eligible_global1_set)
    if keep.any():
        sub = chunk.loc[keep].copy()
        sub["pos"] = pd.to_numeric(sub["pos"], errors="coerce")
        sub = sub.dropna(subset=["chrom", "pos"]).copy()
        sub["pos"] = sub["pos"].astype(np.int64)
        mapped_chunks.append(sub)

if len(mapped_chunks) == 0:
    raise ValueError("No eligible reference CpGs mapped to CpG.bed by global CpG index.")

cpg_mapped = pd.concat(mapped_chunks, ignore_index=True)

mapped_sites = cpg_mapped.merge(
    site_fi_df,
    on="global_idx_1based",
    how="inner",
    validate="one_to_one"
).sort_values(["chrom", "pos"]).reset_index(drop=True)

print("Mapped eligible sites:", len(mapped_sites), "of", len(site_fi_df))
display(mapped_sites.head())

if len(mapped_sites) == 0:
    raise ValueError("Mapped eligible sites is empty.")

# Direct global CpG index lookup for read scoring
global1_to_eligible_pos = dict(
    zip(mapped_sites["global_idx_1based"].astype(int), mapped_sites["eligible_pos"].astype(int))
)

eligible_pos_to_chrom = dict(
    zip(mapped_sites["eligible_pos"].astype(int), mapped_sites["chrom"].astype(str))
)
eligible_pos_to_pos = dict(
    zip(mapped_sites["eligible_pos"].astype(int), mapped_sites["pos"].astype(int))
)
eligible_pos_to_global1 = dict(
    zip(mapped_sites["eligible_pos"].astype(int), mapped_sites["global_idx_1based"].astype(int))
)

print("Global CpG lookup size:", len(global1_to_eligible_pos))

# ELOVL2 sanity check
elovl2_sites = mapped_sites[
    (mapped_sites["chrom"] == "chr6") &
    (mapped_sites["pos"] >= ELOVL2_SITE_HG38 - ELOVL2_WINDOW_BP) &
    (mapped_sites["pos"] <= ELOVL2_SITE_HG38 + ELOVL2_WINDOW_BP)
].copy()

print("ELOVL2 eligible CpGs in +/-5kb:", len(elovl2_sites))
if len(elovl2_sites) > 0:
    print("ELOVL2 global CpG index range:",
          int(elovl2_sites["global_idx_1based"].min()),
          int(elovl2_sites["global_idx_1based"].max()))
    display(elovl2_sites.head())

### Build window table

In [ ]:
# ============================================================
# 7) BUILD WINDOW TABLE
# ============================================================
windows = []

for chrom, d in mapped_sites.groupby("chrom", sort=False):
    d = d.sort_values("pos").reset_index(drop=True)

    min_pos = int(d["pos"].min())
    max_pos = int(d["pos"].max())

    starts = np.arange(
        math.floor(min_pos / STEP_BP) * STEP_BP,
        max_pos + 1,
        STEP_BP,
        dtype=np.int64
    )

    pos = d["pos"].to_numpy()
    fi = d["I_avg_uniform_20_100"].to_numpy()
    eligible_pos_arr = d["eligible_pos"].to_numpy()
    global1_arr = d["global_idx_1based"].to_numpy()

    for start in starts:
        end = int(start + WINDOW_BP)

        left = np.searchsorted(pos, start, side="left")
        right = np.searchsorted(pos, end, side="right")

        n = right - left
        if n == 0:
            continue

        fi_slice = fi[left:right]

        windows.append({
            "chrom": chrom,
            "window_start": int(start),
            "window_end": int(end),
            "n_eligible_sites": int(n),
            "sum_fisher_info": float(fi_slice.sum()),
            "mean_fisher_info": float(fi_slice.mean()),
            "max_fisher_info": float(fi_slice.max()),
            "eligible_pos_list": ",".join(map(str, eligible_pos_arr[left:right].astype(int).tolist())),
            "global_idx1_list": ",".join(map(str, global1_arr[left:right].astype(int).tolist())),
            "min_global_idx1": int(global1_arr[left:right].min()),
            "max_global_idx1": int(global1_arr[left:right].max()),
        })

window_df = pd.DataFrame(windows)

if len(window_df) == 0:
    raise ValueError("No windows generated.")

window_df = window_df.sort_values(
    ["sum_fisher_info", "n_eligible_sites", "max_fisher_info"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Number of windows:", len(window_df))
display(window_df.head(20))

plt.figure(figsize=(6.5, 5))
plt.scatter(
    window_df["n_eligible_sites"],
    window_df["sum_fisher_info"],
    alpha=0.25,
    s=10
)
plt.xlabel("Eligible CpGs in window")
plt.ylabel("Summed Fisher information")
plt.title("Window information vs site count")
plt.tight_layout()
plt.show()

window_df.to_csv(f"{OUT_DIR}/all_windows_by_fi_and_site_count.csv", index=False)
print("Saved:", f"{OUT_DIR}/all_windows_by_fi_and_site_count.csv")

### 8b) Plot window table and label ELOVL2 region

In [ ]:
# ============================================================
# 7b) PLOT WINDOW TABLE AND LABEL ELOVL2 REGION
# ============================================================

# Define an ELOVL2-centered window using the same WINDOW_BP
ELOVL2_SITE_HG38 = 11044644
ELOVL2_CHROM = "chr6"

elovl2_start = int(ELOVL2_SITE_HG38 - WINDOW_BP // 2)
elovl2_end = int(ELOVL2_SITE_HG38 + WINDOW_BP // 2)

elovl2_sites = mapped_sites[
    (mapped_sites["chrom"] == ELOVL2_CHROM) &
    (mapped_sites["pos"] >= elovl2_start) &
    (mapped_sites["pos"] <= elovl2_end)
].copy()

if len(elovl2_sites) == 0:
    print("No eligible CpGs found in ELOVL2-centered window.")
    elovl2_n = np.nan
    elovl2_fi = np.nan
else:
    elovl2_n = int(len(elovl2_sites))
    elovl2_fi = float(elovl2_sites["I_avg_uniform_20_100"].sum())

    print("ELOVL2-centered window:")
    print(f"  region: {ELOVL2_CHROM}:{elovl2_start}-{elovl2_end}")
    print(f"  eligible CpGs: {elovl2_n}")
    print(f"  summed FI: {elovl2_fi:.6g}")

plt.figure(figsize=(6.8, 5.2))

# All windows
plt.scatter(
    window_df["n_eligible_sites"],
    window_df["sum_fisher_info"],
    alpha=0.25,
    s=10,
    label="All windows"
)

# ELOVL2-centered window
if np.isfinite(elovl2_n) and np.isfinite(elovl2_fi):
    plt.scatter(
        [elovl2_n],
        [elovl2_fi],
        s=90,
        marker="*",
        edgecolors="black",
        linewidths=0.8,
        label="ELOVL2"
    )

    plt.annotate(
        "ELOVL2",
        xy=(elovl2_n, elovl2_fi),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=10,
        ha="left",
        va="bottom",
        arrowprops=dict(arrowstyle="-", linewidth=0.8)
    )

plt.xlabel("Eligible CpGs in window")
plt.ylabel("Summed Fisher information")
plt.title("Window information vs site count")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

### Select regions

In [ ]:
# ============================================================
# 8) SELECT REGIONS
# ============================================================
def pick_window_near(target_fi=None, target_n=None, min_n=None, max_n=None, label="candidate"):
    d = window_df.copy()

    if min_n is not None:
        d = d[d["n_eligible_sites"] >= min_n]
    if max_n is not None:
        d = d[d["n_eligible_sites"] <= max_n]

    if len(d) == 0:
        return None

    score = np.zeros(len(d), dtype=float)

    if target_fi is not None:
        fi_scale = max(window_df["sum_fisher_info"].std(), 1e-12)
        score += ((d["sum_fisher_info"].to_numpy() - target_fi) / fi_scale) ** 2

    if target_n is not None:
        n_scale = max(window_df["n_eligible_sites"].std(), 1e-12)
        score += ((d["n_eligible_sites"].to_numpy() - target_n) / n_scale) ** 2

    out = d.iloc[int(np.argmin(score))].copy()
    out["region_name"] = label
    return out

selected = []

top = window_df.iloc[0].copy()
top["region_name"] = "top_FI_window"
selected.append(top)

w_007_150 = pick_window_near(
    target_fi=0.007,
    target_n=150,
    min_n=100,
    max_n=200,
    label="FI_0p007_n150_window"
)
if w_007_150 is not None:
    selected.append(w_007_150)

w_over250 = pick_window_near(
    target_fi=None,
    target_n=260,
    min_n=250,
    label="over250_eligible_CpG_window"
)
if w_over250 is not None:
    selected.append(w_over250)

d_many = window_df[window_df["n_eligible_sites"] >= 150].copy()
if len(d_many) > 0:
    q25 = d_many["sum_fisher_info"].quantile(0.25)
    w = d_many.iloc[(d_many["sum_fisher_info"] - q25).abs().argsort().iloc[0]].copy()
    w["region_name"] = "many_CpGs_lower_FI_window"
    selected.append(w)

selected.append(pd.Series({
    "region_name": "ELOVL2_pm5kb",
    "chrom": "chr6",
    "window_start": int(ELOVL2_SITE_HG38 - ELOVL2_WINDOW_BP),
    "window_end": int(ELOVL2_SITE_HG38 + ELOVL2_WINDOW_BP),
    "n_eligible_sites": np.nan,
    "sum_fisher_info": np.nan,
    "mean_fisher_info": np.nan,
    "max_fisher_info": np.nan,
    "eligible_pos_list": "",
    "global_idx1_list": "",
    "min_global_idx1": np.nan,
    "max_global_idx1": np.nan,
}))

regions_df = pd.DataFrame(selected).drop_duplicates(
    subset=["region_name", "chrom", "window_start", "window_end"]
).reset_index(drop=True)

# Recompute exact region stats from mapped_sites
for i, row in regions_df.iterrows():
    chrom = str(row["chrom"])
    start = int(row["window_start"])
    end = int(row["window_end"])

    d = mapped_sites[
        (mapped_sites["chrom"] == chrom) &
        (mapped_sites["pos"] >= start) &
        (mapped_sites["pos"] <= end)
    ].copy()

    regions_df.loc[i, "n_eligible_sites"] = int(len(d))
    regions_df.loc[i, "sum_fisher_info"] = float(d["I_avg_uniform_20_100"].sum()) if len(d) else 0.0
    regions_df.loc[i, "mean_fisher_info"] = float(d["I_avg_uniform_20_100"].mean()) if len(d) else np.nan
    regions_df.loc[i, "max_fisher_info"] = float(d["I_avg_uniform_20_100"].max()) if len(d) else np.nan
    regions_df.loc[i, "eligible_pos_list"] = ",".join(map(str, d["eligible_pos"].astype(int).tolist()))
    regions_df.loc[i, "global_idx1_list"] = ",".join(map(str, d["global_idx_1based"].astype(int).tolist()))

    if len(d) > 0:
        regions_df.loc[i, "min_global_idx1"] = int(d["global_idx_1based"].min())
        regions_df.loc[i, "max_global_idx1"] = int(d["global_idx_1based"].max())

display(regions_df[[
    "region_name", "chrom", "window_start", "window_end",
    "n_eligible_sites", "sum_fisher_info",
    "min_global_idx1", "max_global_idx1"
]])

regions_df.to_csv(f"{OUT_DIR}/selected_regions.csv", index=False)
print("Saved:", f"{OUT_DIR}/selected_regions.csv")

### Build region objects for fast global-index read scoring

In [ ]:
# ============================================================
# 9) BUILD REGION OBJECTS FOR FAST GLOBAL-INDEX READ SCORING
# ============================================================
region_objects = []

for _, row in regions_df.iterrows():
    region_name = str(row["region_name"])
    chrom = str(row["chrom"])
    start = int(row["window_start"])
    end = int(row["window_end"])

    d = mapped_sites[
        (mapped_sites["chrom"] == chrom) &
        (mapped_sites["pos"] >= start) &
        (mapped_sites["pos"] <= end)
    ].copy()

    if len(d) == 0:
        print("Skipping empty region:", region_name)
        continue

    global_to_eligible = dict(
        zip(d["global_idx_1based"].astype(int), d["eligible_pos"].astype(int))
    )

    region_objects.append({
        "region_name": region_name,
        "chrom": chrom,
        "window_start": start,
        "window_end": end,
        "min_global_idx1": int(d["global_idx_1based"].min()),
        "max_global_idx1": int(d["global_idx_1based"].max()),
        "global_to_eligible": global_to_eligible,
        "n_eligible_sites": int(len(d)),
        "sum_fisher_info": float(d["I_avg_uniform_20_100"].sum()),
    })

print("Region objects:")
display(pd.DataFrame([
    {k: v for k, v in r.items() if k != "global_to_eligible"}
    for r in region_objects
]))

### PAT iteration and global-index scoring

In [ ]:
# ============================================================
# 10) PAT ITERATION + GLOBAL-INDEX SCORING
# ============================================================
def iter_pat_rows(gs_pat_path: str):
    raw = download_bytes_from_gcs(gs_pat_path)

    if gs_pat_path.endswith(".gz"):
        fh = io.TextIOWrapper(gzip.GzipFile(fileobj=io.BytesIO(raw)), encoding="utf-8")
    else:
        fh = io.StringIO(raw.decode("utf-8"))

    with fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue

            parts = line.split("\t")
            if len(parts) < 4:
                continue

            chrom = parts[0]

            try:
                start_global_idx1 = int(parts[1])
            except Exception:
                continue

            pattern = parts[2]

            try:
                count = int(parts[3])
            except Exception:
                count = 1

            yield chrom, start_global_idx1, pattern, count

def infer_pat_global_mode(gs_pat_path: str, chrom_for_check="chr6", n_rows=2000):
    starts = []

    for chrom, start_idx, pattern, count in iter_pat_rows(gs_pat_path):
        if chrom != chrom_for_check:
            continue
        starts.append(start_idx)
        if len(starts) >= n_rows:
            break

    if len(starts) == 0:
        return "unknown", pd.Series(dtype=float)

    s = pd.Series(starts)

    # For this dataset, chr6 global CpG indices are around millions,
    # local chromosome CpG indices near ELOVL2 are around 130k.
    mode = "global" if s.median() > 1_000_000 else "local_or_unexpected"
    return mode, s.describe()

def ll_curve_binary(m_use, b0, b1, age_grid, eps):
    u_use = 1.0 - m_use
    out = np.empty(len(age_grid), dtype=np.float64)

    for i, a in enumerate(age_grid):
        p = b0 + b1 * float(a)
        p = np.clip(p, eps, 1.0 - eps)
        out[i] = np.sum(m_use * np.log(p) + u_use * np.log(1.0 - p))

    return out

def score_pat_row_against_region(chrom, start_global_idx1, pattern, count, region):
    if chrom != region["chrom"]:
        return None

    row_end_global_idx1 = start_global_idx1 + len(pattern) - 1

    if row_end_global_idx1 < region["min_global_idx1"]:
        return None
    if start_global_idx1 > region["max_global_idx1"]:
        return None

    global_to_eligible = region["global_to_eligible"]

    idx_hits = []
    m_hits = []

    for j, ch in enumerate(pattern):
        global_idx1 = start_global_idx1 + j

        eligible_pos = global_to_eligible.get(int(global_idx1), None)
        if eligible_pos is None:
            continue

        if ch in ("C", "1"):
            idx_hits.append(int(eligible_pos))
            m_hits.append(1.0)
        elif ch in ("T", "0"):
            idx_hits.append(int(eligible_pos))
            m_hits.append(0.0)
        else:
            continue

    if len(idx_hits) < MIN_CPGS_PER_READ:
        return None

    idx_hits = np.array(idx_hits, dtype=np.int64)
    m_use = np.array(m_hits, dtype=np.float64)

    # Select strongest CpGs among CpGs actually observed on the read.
    order = np.argsort(np.abs(t_stat_eligible[idx_hits]))[::-1]
    order = order[:min(K_INFER, len(order))]

    idx_use = idx_hits[order]
    m_use = m_use[order]

    if len(idx_use) < MIN_CPGS_PER_READ:
        return None

    b0 = intercept_eligible[idx_use]
    b1 = coef_eligible[idx_use]

    ll = ll_curve_binary(m_use, b0, b1, age_grid_ref, EPS)
    pred_age = float(age_grid_ref[np.argmax(ll)])

    n_c = int(np.sum(m_use == 1.0))
    n_t = int(np.sum(m_use == 0.0))
    meth_frac = float(n_c / (n_c + n_t)) if (n_c + n_t) > 0 else np.nan

    return {
        "region_name": region["region_name"],
        "chrom": chrom,
        "start_global_idx1": int(start_global_idx1),
        "end_global_idx1": int(row_end_global_idx1),
        "count": int(count),
        "pred_age": pred_age,
        "n_cpgs_used": int(len(idx_use)),
        "n_methylated_C": n_c,
        "n_unmethylated_T": n_t,
        "meth_frac": meth_frac,
        "is_fully_methylated": bool(n_t == 0 and n_c > 0),
        "is_fully_unmethylated": bool(n_c == 0 and n_t > 0),
        "eligible_pos_used": ",".join(map(str, idx_use.astype(int).tolist())),
        "pattern": pattern,
    }

### One-sample smoke test before full run

In [ ]:
# ============================================================
# 11) ONE-SAMPLE SMOKE TEST BEFORE FULL RUN
# ============================================================
one = test_manifest_run.iloc[0]
one_pat = resolve_pat_path_from_beta(one["bulk_beta_path"])

print("person_id:", one["person_id"])
print("true age:", one["Age"])
print("pat:", one_pat)

mode, desc = infer_pat_global_mode(one_pat)
print("PAT index mode check:", mode)
display(desc)

if mode != "global":
    raise ValueError(
        "This notebook expects global CpG-index PAT files. "
        "The first sample did not look global."
    )

smoke_rows = []

for chrom, start_global_idx1, pattern, count in tqdm(iter_pat_rows(one_pat), desc="Smoke sample PAT rows"):
    for region in region_objects:
        scored = score_pat_row_against_region(
            chrom=chrom,
            start_global_idx1=start_global_idx1,
            pattern=pattern,
            count=count,
            region=region
        )
        if scored is None:
            continue

        scored["person_id"] = str(one["person_id"])
        scored["true_age"] = float(one["Age"])
        scored["residual"] = scored["pred_age"] - float(one["Age"])
        scored["pat_path"] = one_pat
        smoke_rows.append(scored)

smoke_df = pd.DataFrame(smoke_rows)

print("Smoke test rows:", smoke_df.shape)

if len(smoke_df) == 0:
    print("No scored reads. Running raw hit diagnostic next.")
else:
    display(
        smoke_df.groupby("region_name").agg(
            n_pattern_rows=("pred_age", "size"),
            weighted_reads=("count", "sum"),
            mean_n_cpgs=("n_cpgs_used", "mean"),
            median_pred=("pred_age", "median"),
            mean_meth_frac=("meth_frac", "mean"),
            frac_fully_methylated=("is_fully_methylated", "mean"),
            frac_fully_unmethylated=("is_fully_unmethylated", "mean"),
        ).reset_index()
    )
    display(smoke_df.head())

### Raw hit diagnostic if smoke test is zero

In [ ]:
# ============================================================
# 12) RAW HIT DIAGNOSTIC IF SMOKE TEST IS ZERO
# ============================================================
def raw_hit_diagnostic_one_sample(gs_pat_path, region_objects):
    rows = []

    for chrom, start_global_idx1, pattern, count in tqdm(iter_pat_rows(gs_pat_path), desc="Raw hit diagnostic"):
        row_end = start_global_idx1 + len(pattern) - 1

        for region in region_objects:
            if chrom != region["chrom"]:
                continue
            if row_end < region["min_global_idx1"]:
                continue
            if start_global_idx1 > region["max_global_idx1"]:
                continue

            n_eligible_seen = 0
            n_informative = 0
            n_c = 0
            n_t = 0

            global_to_eligible = region["global_to_eligible"]

            for j, ch in enumerate(pattern):
                global_idx1 = start_global_idx1 + j
                if global_idx1 not in global_to_eligible:
                    continue

                n_eligible_seen += 1

                if ch in ("C", "1"):
                    n_informative += 1
                    n_c += 1
                elif ch in ("T", "0"):
                    n_informative += 1
                    n_t += 1

            if n_eligible_seen > 0:
                rows.append({
                    "region_name": region["region_name"],
                    "chrom": chrom,
                    "start_global_idx1": start_global_idx1,
                    "end_global_idx1": row_end,
                    "count": count,
                    "n_eligible_seen": n_eligible_seen,
                    "n_informative": n_informative,
                    "n_C": n_c,
                    "n_T": n_t,
                })

    return pd.DataFrame(rows)

if len(smoke_df) == 0:
    hit_diag_df = raw_hit_diagnostic_one_sample(one_pat, region_objects)
    print("Raw hit diagnostic rows:", hit_diag_df.shape)

    if len(hit_diag_df) > 0:
        display(
            hit_diag_df.groupby("region_name").agg(
                rows_with_any_eligible=("n_eligible_seen", "size"),
                max_eligible_seen=("n_eligible_seen", "max"),
                max_informative=("n_informative", "max"),
                rows_ge_1=("n_informative", lambda x: int((x >= 1).sum())),
                rows_ge_3=("n_informative", lambda x: int((x >= 3).sum())),
                rows_ge_5=("n_informative", lambda x: int((x >= 5).sum())),
            ).reset_index()
        )
        display(hit_diag_df.sort_values("n_informative", ascending=False).head(30))
    else:
        print("No raw eligible CpG overlaps in selected regions for this sample.")

### Full run across test samples

In [ ]:
# ============================================================
# 13) FULL RUN ACROSS TEST SAMPLES
# ============================================================
if len(smoke_df) == 0:
    raise ValueError("Smoke test produced zero reads. Do not run full analysis yet.")

all_read_rows = []
sample_region_rows = []

for sample in tqdm(test_manifest_run.itertuples(index=False), total=len(test_manifest_run), desc="Samples"):
    person_id = str(sample.person_id)
    true_age = float(sample.Age)
    beta_path = sample.bulk_beta_path
    pat_path = resolve_pat_path_from_beta(beta_path)

    if pat_path is None:
        print("PAT not found:", person_id, beta_path)
        continue

    mode, _ = infer_pat_global_mode(pat_path)
    if mode != "global":
        print("Skipping non-global/unexpected PAT:", person_id, mode)
        continue

    sample_reads = []

    for chrom, start_global_idx1, pattern, count in iter_pat_rows(pat_path):
        for region in region_objects:
            scored = score_pat_row_against_region(
                chrom=chrom,
                start_global_idx1=start_global_idx1,
                pattern=pattern,
                count=count,
                region=region
            )

            if scored is None:
                continue

            scored["person_id"] = person_id
            scored["true_age"] = true_age
            scored["pat_path"] = pat_path
            scored["residual"] = scored["pred_age"] - true_age

            all_read_rows.append(scored)
            sample_reads.append(scored)

    if len(sample_reads) == 0:
        continue

    d_sample = pd.DataFrame(sample_reads)

    for region_name, d in d_sample.groupby("region_name"):
        w = d["count"].to_numpy(dtype=float)
        x = d["pred_age"].to_numpy(dtype=float)

        weighted_mean = np.average(x, weights=w)

        ord_idx = np.argsort(x)
        x_ord = x[ord_idx]
        w_ord = w[ord_idx]
        cumw = np.cumsum(w_ord) / np.sum(w_ord)
        weighted_median = x_ord[np.searchsorted(cumw, 0.5)]

        sample_region_rows.append({
            "person_id": person_id,
            "true_age": true_age,
            "region_name": region_name,
            "n_pattern_rows": int(len(d)),
            "n_weighted_reads": int(w.sum()),
            "weighted_mean_pred": float(weighted_mean),
            "weighted_median_pred": float(weighted_median),
            "weighted_mean_residual": float(weighted_mean - true_age),
            "weighted_median_residual": float(weighted_median - true_age),
            "mean_n_cpgs_used": float(np.average(d["n_cpgs_used"], weights=w)),
            "mean_meth_frac": float(np.average(d["meth_frac"], weights=w)),
            "frac_fully_methylated": float(np.average(d["is_fully_methylated"].astype(float), weights=w)),
            "frac_fully_unmethylated": float(np.average(d["is_fully_unmethylated"].astype(float), weights=w)),
        })

reads_df = pd.DataFrame(all_read_rows)
sample_region_df = pd.DataFrame(sample_region_rows)

print("reads_df:", reads_df.shape)
print("sample_region_df:", sample_region_df.shape)

if len(reads_df) == 0:
    raise ValueError("Full run produced zero read-level predictions.")

display(reads_df.head())
display(sample_region_df.head())

reads_df.to_csv(f"{OUT_DIR}/read_level_predictions.csv", index=False)
sample_region_df.to_csv(f"{OUT_DIR}/sample_region_predictions.csv", index=False)

print("Saved:", f"{OUT_DIR}/read_level_predictions.csv")
print("Saved:", f"{OUT_DIR}/sample_region_predictions.csv")

### Region-level performance summary

In [ ]:
# ============================================================
# 14) REGION-LEVEL PERFORMANCE SUMMARY
# ============================================================
def corr_safe(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)

    if ok.sum() < 3:
        return np.nan
    if np.std(x[ok]) == 0 or np.std(y[ok]) == 0:
        return np.nan

    return float(np.corrcoef(x[ok], y[ok])[0, 1])

summary_rows = []

for region_name, d in sample_region_df.groupby("region_name"):
    err_med = d["weighted_median_pred"] - d["true_age"]
    err_mean = d["weighted_mean_pred"] - d["true_age"]

    summary_rows.append({
        "region_name": region_name,
        "n_samples_ok": int(len(d)),
        "total_weighted_reads": int(d["n_weighted_reads"].sum()),
        "median_pred_MAE": float(np.mean(np.abs(err_med))),
        "mean_pred_MAE": float(np.mean(np.abs(err_mean))),
        "median_pred_bias": float(np.mean(err_med)),
        "mean_pred_bias": float(np.mean(err_mean)),
        "median_pred_corr": corr_safe(d["weighted_median_pred"], d["true_age"]),
        "mean_pred_corr": corr_safe(d["weighted_mean_pred"], d["true_age"]),
        "median_reads_per_sample": float(d["n_weighted_reads"].median()),
        "median_CpGs_per_read": float(d["mean_n_cpgs_used"].median()),
        "median_meth_frac": float(d["mean_meth_frac"].median()),
        "median_frac_fully_methylated": float(d["frac_fully_methylated"].median()),
        "median_frac_fully_unmethylated": float(d["frac_fully_unmethylated"].median()),
    })

region_summary_df = pd.DataFrame(summary_rows).sort_values("median_pred_MAE")
display(region_summary_df)

region_summary_df.to_csv(f"{OUT_DIR}/region_summary.csv", index=False)
print("Saved:", f"{OUT_DIR}/region_summary.csv")

### Predicted vs true age plots

In [ ]:
# ============================================================
# 15) PREDICTED VS TRUE PLOTS
# ============================================================
for region_name, d in sample_region_df.groupby("region_name"):
    if len(d) < 3:
        continue

    x = d["true_age"].to_numpy()
    y = d["weighted_median_pred"].to_numpy()

    lo = min(np.nanmin(x), np.nanmin(y))
    hi = max(np.nanmax(x), np.nanmax(y))

    plt.figure(figsize=(5.5, 5.5))
    plt.scatter(x, y, alpha=0.75)
    plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.5)
    plt.xlabel("True age")
    plt.ylabel("Weighted median read-predicted age")
    plt.title(region_name)
    plt.tight_layout()
    plt.show()

### ELOVL2 boundary methylation diagnostic

In [ ]:
# ============================================================
# 16) ELOVL2 BOUNDARY METHYLATION DIAGNOSTIC
# ============================================================
def classify_prediction_band(d):
    d = d.copy()

    low_cut = d["pred_age"].quantile(BOUNDARY_Q_LOW)
    high_cut = d["pred_age"].quantile(BOUNDARY_Q_HIGH)

    d["prediction_band"] = "middle"
    d.loc[d["pred_age"] <= low_cut, "prediction_band"] = "low_boundary"
    d.loc[d["pred_age"] >= high_cut, "prediction_band"] = "high_boundary"

    return d, low_cut, high_cut

if "ELOVL2_pm5kb" not in reads_df["region_name"].unique():
    print("No ELOVL2_pm5kb reads found.")
else:
    elovl2_reads = reads_df[reads_df["region_name"] == "ELOVL2_pm5kb"].copy()
    elovl2_reads, low_cut, high_cut = classify_prediction_band(elovl2_reads)

    print("ELOVL2 read rows:", len(elovl2_reads))
    print("Low boundary cutoff:", low_cut)
    print("High boundary cutoff:", high_cut)

    boundary_summary = (
        elovl2_reads
        .groupby("prediction_band")
        .apply(lambda d: pd.Series({
            "n_pattern_rows": int(len(d)),
            "weighted_reads": int(d["count"].sum()),
            "mean_pred_age": float(np.average(d["pred_age"], weights=d["count"])),
            "median_pred_age": float(d["pred_age"].median()),
            "mean_true_age": float(np.average(d["true_age"], weights=d["count"])),
            "mean_n_cpgs_used": float(np.average(d["n_cpgs_used"], weights=d["count"])),
            "mean_meth_frac": float(np.average(d["meth_frac"], weights=d["count"])),
            "frac_fully_methylated": float(np.average(d["is_fully_methylated"].astype(float), weights=d["count"])),
            "frac_fully_unmethylated": float(np.average(d["is_fully_unmethylated"].astype(float), weights=d["count"])),
        }))
        .reset_index()
    )

    display(boundary_summary)

    plt.figure(figsize=(6, 5))
    plt.scatter(
        elovl2_reads["meth_frac"],
        elovl2_reads["pred_age"],
        s=np.clip(elovl2_reads["count"].to_numpy() * 8, 8, 60),
        alpha=0.35
    )
    plt.axhline(low_cut, linestyle="--", linewidth=1.2)
    plt.axhline(high_cut, linestyle="--", linewidth=1.2)
    plt.xlabel("Read methylation fraction across eligible ELOVL2 CpGs")
    plt.ylabel("Single-read predicted age")
    plt.title("ELOVL2: predicted age vs read methylation fraction")
    plt.tight_layout()
    plt.show()

    for band in ["low_boundary", "middle", "high_boundary"]:
        d = elovl2_reads[elovl2_reads["prediction_band"] == band]
        if len(d) == 0:
            continue

        plt.figure(figsize=(6, 4))
        plt.hist(d["meth_frac"], bins=25, weights=d["count"], alpha=0.8)
        plt.xlabel("Read methylation fraction")
        plt.ylabel("Weighted read count")
        plt.title(f"ELOVL2 methylation fraction: {band}")
        plt.tight_layout()
        plt.show()

    example_cols = [
        "person_id", "true_age", "pred_age", "prediction_band",
        "n_cpgs_used", "n_methylated_C", "n_unmethylated_T",
        "meth_frac", "is_fully_methylated", "is_fully_unmethylated",
        "eligible_pos_used", "pattern"
    ]

    examples = []
    for band in ["low_boundary", "middle", "high_boundary"]:
        d = elovl2_reads[elovl2_reads["prediction_band"] == band].copy()
        if len(d) == 0:
            continue
        examples.append(d.sort_values("n_cpgs_used", ascending=False).head(10)[example_cols])

    if len(examples) > 0:
        example_df = pd.concat(examples, ignore_index=True)
        display(example_df)

    elovl2_reads.to_csv(f"{OUT_DIR}/elovl2_read_boundary_diagnostics.csv", index=False)
    boundary_summary.to_csv(f"{OUT_DIR}/elovl2_boundary_summary.csv", index=False)

    print("Saved:", f"{OUT_DIR}/elovl2_read_boundary_diagnostics.csv")
    print("Saved:", f"{OUT_DIR}/elovl2_boundary_summary.csv")

### All-region boundary diagnostic

In [ ]:
# ============================================================
# 17) ALL-REGION BOUNDARY DIAGNOSTIC
# ============================================================
diagnostic_rows = []

for region_name, d0 in reads_df.groupby("region_name"):
    d, low_cut, high_cut = classify_prediction_band(d0)

    for band, db in d.groupby("prediction_band"):
        diagnostic_rows.append({
            "region_name": region_name,
            "prediction_band": band,
            "low_cut": float(low_cut),
            "high_cut": float(high_cut),
            "n_pattern_rows": int(len(db)),
            "weighted_reads": int(db["count"].sum()),
            "mean_pred_age": float(np.average(db["pred_age"], weights=db["count"])),
            "mean_true_age": float(np.average(db["true_age"], weights=db["count"])),
            "mean_n_cpgs_used": float(np.average(db["n_cpgs_used"], weights=db["count"])),
            "mean_meth_frac": float(np.average(db["meth_frac"], weights=db["count"])),
            "frac_fully_methylated": float(np.average(db["is_fully_methylated"].astype(float), weights=db["count"])),
            "frac_fully_unmethylated": float(np.average(db["is_fully_unmethylated"].astype(float), weights=db["count"])),
        })

all_boundary_diag_df = pd.DataFrame(diagnostic_rows)
display(all_boundary_diag_df.sort_values(["region_name", "prediction_band"]))

all_boundary_diag_df.to_csv(f"{OUT_DIR}/boundary_methylation_diagnostics_all_regions.csv", index=False)
print("Saved:", f"{OUT_DIR}/boundary_methylation_diagnostics_all_regions.csv")

### Pick representative samples for read-level distribution plots

In [ ]:
# ============================================================
# 19) PICK REPRESENTATIVE SAMPLES FOR READ-LEVEL DISTRIBUTION PLOTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use regions in the order selected earlier
REGION_ORDER = regions_df["region_name"].tolist()

# Choose a few representative samples across the age range among samples with reads
sample_age_df = (
    sample_region_df[["person_id", "true_age"]]
    .drop_duplicates()
    .sort_values("true_age")
    .reset_index(drop=True)
)

N_EXAMPLE_SAMPLES = 4

if len(sample_age_df) <= N_EXAMPLE_SAMPLES:
    example_people = sample_age_df["person_id"].tolist()
else:
    idx = np.linspace(0, len(sample_age_df) - 1, N_EXAMPLE_SAMPLES).round().astype(int)
    example_people = sample_age_df.iloc[idx]["person_id"].tolist()

print("Example people:")
display(sample_age_df[sample_age_df["person_id"].isin(example_people)])

### Helper functions for weighted summary and histogram plots

In [ ]:
# ============================================================
# 20) HELPER FUNCTIONS FOR WEIGHTED SUMMARY AND HISTOGRAM PLOTS
# ============================================================
def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[ok]
    weights = weights[ok]

    if len(values) == 0:
        return np.nan

    order = np.argsort(values)
    values = values[order]
    weights = weights[order]

    cumw = np.cumsum(weights) / np.sum(weights)
    return float(values[np.searchsorted(cumw, 0.5)])


def summarize_read_predictions(d):
    w = d["count"].to_numpy(dtype=float)
    x = d["pred_age"].to_numpy(dtype=float)

    return {
        "n_pattern_rows": int(len(d)),
        "weighted_reads": int(w.sum()),
        "weighted_mean_pred": float(np.average(x, weights=w)),
        "weighted_median_pred": weighted_median(x, w),
        "mean_meth_frac": float(np.average(d["meth_frac"], weights=w)),
        "mean_n_cpgs_used": float(np.average(d["n_cpgs_used"], weights=w)),
    }


def plot_read_age_histogram_for_sample_region(
    reads_df,
    person_id,
    region_name,
    bins=np.arange(0, 101, 2),
    xlim=(0, 100)
):
    d = reads_df[
        (reads_df["person_id"].astype(str) == str(person_id)) &
        (reads_df["region_name"] == region_name)
    ].copy()

    if len(d) == 0:
        print(f"No reads for person_id={person_id}, region={region_name}")
        return None

    true_age = float(d["true_age"].iloc[0])
    s = summarize_read_predictions(d)

    plt.figure(figsize=(7, 4.5))
    plt.hist(
        d["pred_age"].to_numpy(dtype=float),
        bins=bins,
        weights=d["count"].to_numpy(dtype=float),
        alpha=0.75
    )

    plt.axvline(true_age, linestyle="--", linewidth=2, label=f"True age = {true_age:.1f}")
    plt.axvline(s["weighted_mean_pred"], linestyle="-", linewidth=2, label=f"Mean = {s['weighted_mean_pred']:.1f}")
    plt.axvline(s["weighted_median_pred"], linestyle=":", linewidth=2.5, label=f"Median = {s['weighted_median_pred']:.1f}")

    plt.xlim(*xlim)
    plt.xlabel("Read-level predicted age")
    plt.ylabel("Weighted read count")
    plt.title(
        f"{region_name}\n"
        f"person_id={person_id}, n reads={s['weighted_reads']}, "
        f"mean CpGs/read={s['mean_n_cpgs_used']:.1f}"
    )
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    return s

### Read-level histograms for a few samples across all 5 regions

In [ ]:
# ============================================================
# 21) READ-LEVEL HISTOGRAMS FOR A FEW SAMPLES ACROSS ALL 5 REGIONS
# ============================================================

plot_summaries = []

for person_id in example_people:
    print("=" * 90)
    print("person_id:", person_id)

    for region_name in REGION_ORDER:
        s = plot_read_age_histogram_for_sample_region(
            reads_df=reads_df,
            person_id=person_id,
            region_name=region_name,
            bins=np.arange(0, 101, 2),
            xlim=(0, 100)
        )

        if s is not None:
            s["person_id"] = str(person_id)
            s["region_name"] = region_name
            s["true_age"] = float(
                reads_df.loc[
                    (reads_df["person_id"].astype(str) == str(person_id)) &
                    (reads_df["region_name"] == region_name),
                    "true_age"
                ].iloc[0]
            )
            plot_summaries.append(s)

plot_summary_df = pd.DataFrame(plot_summaries)
display(plot_summary_df[[
    "person_id", "region_name", "true_age",
    "n_pattern_rows", "weighted_reads",
    "weighted_mean_pred", "weighted_median_pred",
    "mean_n_cpgs_used", "mean_meth_frac"
]])

### Compact multi-panel histograms: one sample, all regions

In [ ]:
# ============================================================
# 22) COMPACT MULTI-PANEL HISTOGRAMS: ONE SAMPLE, ALL REGIONS
# ============================================================

def plot_all_regions_for_one_sample(person_id, reads_df, region_order=REGION_ORDER):
    person_id = str(person_id)

    d_person = reads_df[reads_df["person_id"].astype(str) == person_id].copy()
    if len(d_person) == 0:
        print("No reads for person:", person_id)
        return

    true_age = float(d_person["true_age"].iloc[0])

    n_regions = len(region_order)
    ncols = 2
    nrows = int(np.ceil(n_regions / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows), sharex=True)
    axes = np.ravel(axes)

    for ax_i, region_name in enumerate(region_order):
        ax = axes[ax_i]

        d = d_person[d_person["region_name"] == region_name].copy()

        if len(d) == 0:
            ax.text(0.5, 0.5, "No reads", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(region_name)
            continue

        s = summarize_read_predictions(d)

        ax.hist(
            d["pred_age"].to_numpy(dtype=float),
            bins=np.arange(0, 101, 2),
            weights=d["count"].to_numpy(dtype=float),
            alpha=0.75
        )

        ax.axvline(true_age, linestyle="--", linewidth=2, label=f"True {true_age:.1f}")
        ax.axvline(s["weighted_mean_pred"], linestyle="-", linewidth=2, label=f"Mean {s['weighted_mean_pred']:.1f}")
        ax.axvline(s["weighted_median_pred"], linestyle=":", linewidth=2.5, label=f"Median {s['weighted_median_pred']:.1f}")

        ax.set_xlim(0, 100)
        ax.set_title(
            f"{region_name}\n"
            f"reads={s['weighted_reads']}, CpGs/read={s['mean_n_cpgs_used']:.1f}, meth={s['mean_meth_frac']:.2f}"
        )
        ax.set_xlabel("Read-level predicted age")
        ax.set_ylabel("Weighted read count")
        ax.legend(frameon=False, fontsize=9)

    for j in range(n_regions, len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"Read-level predicted-age distributions, person_id={person_id}", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()


for person_id in example_people:
    plot_all_regions_for_one_sample(person_id, reads_df)

### ELOVL2 extreme boundary read examples

In [ ]:
# ============================================================
# 23) ELOVL2 EXTREME BOUNDARY READ EXAMPLES
# ============================================================

ELOVL2_REGION_NAME = "ELOVL2_pm5kb"

elovl2_reads = reads_df[reads_df["region_name"] == ELOVL2_REGION_NAME].copy()

if len(elovl2_reads) == 0:
    raise ValueError("No ELOVL2 reads found in reads_df.")

low_cut = elovl2_reads["pred_age"].quantile(BOUNDARY_Q_LOW)
high_cut = elovl2_reads["pred_age"].quantile(BOUNDARY_Q_HIGH)

elovl2_reads["prediction_band"] = "middle"
elovl2_reads.loc[elovl2_reads["pred_age"] <= low_cut, "prediction_band"] = "low_boundary"
elovl2_reads.loc[elovl2_reads["pred_age"] >= high_cut, "prediction_band"] = "high_boundary"

print("ELOVL2 rows:", len(elovl2_reads))
print("Low boundary cutoff:", low_cut)
print("High boundary cutoff:", high_cut)

boundary_summary = (
    elovl2_reads
    .groupby("prediction_band")
    .apply(lambda d: pd.Series({
        "n_pattern_rows": int(len(d)),
        "weighted_reads": int(d["count"].sum()),
        "mean_pred_age": float(np.average(d["pred_age"], weights=d["count"])),
        "median_pred_age": float(d["pred_age"].median()),
        "mean_true_age": float(np.average(d["true_age"], weights=d["count"])),
        "mean_n_cpgs_used": float(np.average(d["n_cpgs_used"], weights=d["count"])),
        "mean_meth_frac": float(np.average(d["meth_frac"], weights=d["count"])),
        "frac_fully_methylated": float(np.average(d["is_fully_methylated"].astype(float), weights=d["count"])),
        "frac_fully_unmethylated": float(np.average(d["is_fully_unmethylated"].astype(float), weights=d["count"])),
    }))
    .reset_index()
)

display(boundary_summary)

### Table of lowest and highest ELOVL2 reads

In [ ]:
# ============================================================
# 24) TABLE OF LOWEST AND HIGHEST ELOVL2 READS
# ============================================================

example_cols = [
    "person_id", "true_age", "pred_age", "prediction_band",
    "start_global_idx1", "end_global_idx1",
    "n_cpgs_used", "n_methylated_C", "n_unmethylated_T",
    "meth_frac", "is_fully_methylated", "is_fully_unmethylated",
    "eligible_pos_used", "pattern"
]

LOW_N = 15
HIGH_N = 15

lowest_reads = (
    elovl2_reads
    .sort_values(["pred_age", "n_cpgs_used"], ascending=[True, False])
    .head(LOW_N)
    .copy()
)

highest_reads = (
    elovl2_reads
    .sort_values(["pred_age", "n_cpgs_used"], ascending=[False, False])
    .head(HIGH_N)
    .copy()
)

extreme_elovl2_reads = pd.concat([lowest_reads, highest_reads], ignore_index=True)

display(extreme_elovl2_reads[example_cols])

extreme_elovl2_reads[example_cols].to_csv(
    f"{OUT_DIR}/elovl2_extreme_boundary_read_examples.csv",
    index=False
)

print("Saved:", f"{OUT_DIR}/elovl2_extreme_boundary_read_examples.csv")

### Compact C/T signatures for extreme ELOVL2 reads

In [ ]:
# ============================================================
# 27) COMPACT C/T SIGNATURES FOR EXTREME ELOVL2 READS
# ============================================================

def extract_used_ct_signature(row):
    """
    Return a compact representation of the read's methylation calls.
    This uses n_methylated_C and n_unmethylated_T for summary, plus raw pattern preview.
    The exact eligible CpG order is stored in eligible_pos_used.
    """
    n_c = int(row["n_methylated_C"])
    n_t = int(row["n_unmethylated_T"])
    return "C" * n_c + "T" * n_t

extreme_show = extreme_elovl2_reads.copy()
extreme_show["compact_CT_signature_unordered"] = extreme_show.apply(extract_used_ct_signature, axis=1)
extreme_show["pattern_preview"] = extreme_show["pattern"].astype(str).str.slice(0, 120)

display(extreme_show[[
    "person_id", "true_age", "pred_age", "prediction_band",
    "n_cpgs_used", "n_methylated_C", "n_unmethylated_T",
    "meth_frac", "compact_CT_signature_unordered",
    "pattern_preview",
    "eligible_pos_used"
]])

### Prep ELOVL2 region + ordered eligible CpGs

In [ ]:
# ============================================================
# 28) PREP ELOVL2 REGION + ORDERED ELIGIBLE CpGs
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ELOVL2_REGION_NAME = "ELOVL2_pm5kb"

# region object
elovl2_region = None
for r in region_objects:
    if r["region_name"] == ELOVL2_REGION_NAME:
        elovl2_region = r
        break

if elovl2_region is None:
    raise ValueError("Could not find ELOVL2 region object.")

# ordered eligible CpGs in this region
elovl2_sites_df = mapped_sites[
    (mapped_sites["chrom"] == elovl2_region["chrom"]) &
    (mapped_sites["pos"] >= elovl2_region["window_start"]) &
    (mapped_sites["pos"] <= elovl2_region["window_end"])
].copy().sort_values("pos").reset_index(drop=True)

if len(elovl2_sites_df) == 0:
    raise ValueError("No eligible CpGs found in ELOVL2 region.")

eligible_pos_order = elovl2_sites_df["eligible_pos"].astype(int).tolist()
eligible_pos_to_col = {ep: i for i, ep in enumerate(eligible_pos_order)}
eligible_pos_to_genomic_pos = dict(
    zip(elovl2_sites_df["eligible_pos"].astype(int), elovl2_sites_df["pos"].astype(int))
)

print("Number of eligible ELOVL2 CpGs:", len(elovl2_sites_df))
display(elovl2_sites_df[["chrom", "pos", "global_idx_1based", "eligible_pos", "t_stat", "I_avg_uniform_20_100"]])

### Reconstruct the used C/T calls for one read

In [ ]:
# ============================================================
# 29) RECONSTRUCT THE USED C/T CALLS FOR ONE READ
# ============================================================
def reconstruct_read_vector_from_row(row, region, eligible_pos_to_col, use_only_predicted_sites=True):
    """
    Returns a vector across all eligible CpGs in the region:
      1 = methylated
      0 = unmethylated
      np.nan = not covered / not used
    """
    vec = np.full(len(eligible_pos_to_col), np.nan, dtype=float)

    used_set = None
    if use_only_predicted_sites:
        used_txt = str(row["eligible_pos_used"])
        used_set = set(int(x) for x in used_txt.split(",") if x != "")

    start_global_idx1 = int(row["start_global_idx1"])
    pattern = str(row["pattern"])
    global_to_eligible = region["global_to_eligible"]

    for j, ch in enumerate(pattern):
        gidx = start_global_idx1 + j
        eligible_pos = global_to_eligible.get(gidx, None)
        if eligible_pos is None:
            continue

        if use_only_predicted_sites and (eligible_pos not in used_set):
            continue

        col = eligible_pos_to_col.get(int(eligible_pos), None)
        if col is None:
            continue

        if ch in ("C", "1"):
            vec[col] = 1.0
        elif ch in ("T", "0"):
            vec[col] = 0.0
        else:
            continue

    return vec

### Choose low-boundary and high-boundary reads to show

In [ ]:
# ============================================================
# 30) CHOOSE A FEW LOW-BOUNDARY AND HIGH-BOUNDARY READS TO SHOW
# ============================================================
N_SHOW_PER_SIDE = 8

# if not already present, define boundaries again
elovl2_reads_all = reads_df[reads_df["region_name"] == ELOVL2_REGION_NAME].copy()

low_cut = elovl2_reads_all["pred_age"].quantile(BOUNDARY_Q_LOW)
high_cut = elovl2_reads_all["pred_age"].quantile(BOUNDARY_Q_HIGH)

elovl2_reads_all["prediction_band"] = "middle"
elovl2_reads_all.loc[elovl2_reads_all["pred_age"] <= low_cut, "prediction_band"] = "low_boundary"
elovl2_reads_all.loc[elovl2_reads_all["pred_age"] >= high_cut, "prediction_band"] = "high_boundary"

lowest_reads_show = (
    elovl2_reads_all[elovl2_reads_all["prediction_band"] == "low_boundary"]
    .sort_values(["pred_age", "n_cpgs_used"], ascending=[True, False])
    .head(N_SHOW_PER_SIDE)
    .copy()
)

highest_reads_show = (
    elovl2_reads_all[elovl2_reads_all["prediction_band"] == "high_boundary"]
    .sort_values(["pred_age", "n_cpgs_used"], ascending=[False, False])
    .head(N_SHOW_PER_SIDE)
    .copy()
)

print("Lowest boundary reads selected:", len(lowest_reads_show))
print("Highest boundary reads selected:", len(highest_reads_show))

display(lowest_reads_show[[
    "person_id", "true_age", "pred_age", "n_cpgs_used",
    "n_methylated_C", "n_unmethylated_T", "meth_frac"
]])

display(highest_reads_show[[
    "person_id", "true_age", "pred_age", "n_cpgs_used",
    "n_methylated_C", "n_unmethylated_T", "meth_frac"
]])

### Plot read x CpG matrix for lowest / highest ELOVL2 reads

In [ ]:
# ============================================================
# 31) PLOT READ x CpG MATRIX FOR LOWEST / HIGHEST ELOVL2 READS
# ============================================================
def build_matrix_for_plot(df_show, region, eligible_pos_to_col, use_only_predicted_sites=True):
    mats = []
    labels = []

    for _, row in df_show.iterrows():
        vec = reconstruct_read_vector_from_row(
            row=row,
            region=region,
            eligible_pos_to_col=eligible_pos_to_col,
            use_only_predicted_sites=use_only_predicted_sites
        )
        mats.append(vec)

        labels.append(
            f"id={row['person_id']} | pred={row['pred_age']:.1f} | true={row['true_age']:.1f} | "
            f"meth={row['meth_frac']:.2f} | n={int(row['n_cpgs_used'])}"
        )

    if len(mats) == 0:
        return None, None

    return np.vstack(mats), labels


def plot_read_matrix(mat, labels, title, genomic_positions, figsize_scale=0.45):
    if mat is None or len(mat) == 0:
        print("Nothing to plot for:", title)
        return

    cmap = plt.cm.RdBu_r.copy()
    cmap.set_bad("lightgray")

    fig_h = max(4, 0.55 * mat.shape[0])
    fig_w = max(9, figsize_scale * mat.shape[1])

    plt.figure(figsize=(fig_w, fig_h))
    im = plt.imshow(mat, aspect="auto", interpolation="nearest", cmap=cmap, vmin=0, vmax=1)

    plt.yticks(np.arange(len(labels)), labels, fontsize=8)

    xticks = np.arange(len(genomic_positions))
    xtick_labels = [str(x) for x in genomic_positions]

    # reduce clutter
    if len(xticks) > 12:
        keep = np.linspace(0, len(xticks) - 1, 12).round().astype(int)
        plt.xticks(keep, [xtick_labels[i] for i in keep], rotation=90, fontsize=8)
    else:
        plt.xticks(xticks, xtick_labels, rotation=90, fontsize=8)

    plt.xlabel("Eligible ELOVL2 CpG genomic position")
    plt.ylabel("Reads")
    plt.title(title)

    cbar = plt.colorbar(im)
    cbar.set_label("Methylation state (0 = T/unmethylated, 1 = C/methylated)")

    plt.tight_layout()
    plt.show()


genomic_positions = [eligible_pos_to_genomic_pos[ep] for ep in eligible_pos_order]

mat_low, labels_low = build_matrix_for_plot(
    lowest_reads_show,
    region=elovl2_region,
    eligible_pos_to_col=eligible_pos_to_col,
    use_only_predicted_sites=True
)

mat_high, labels_high = build_matrix_for_plot(
    highest_reads_show,
    region=elovl2_region,
    eligible_pos_to_col=eligible_pos_to_col,
    use_only_predicted_sites=True
)

plot_read_matrix(
    mat_low,
    labels_low,
    title="ELOVL2 lowest-boundary reads",
    genomic_positions=genomic_positions
)

plot_read_matrix(
    mat_high,
    labels_high,
    title="ELOVL2 highest-boundary reads",
    genomic_positions=genomic_positions
)

### Optional: show full ELOVL2 coverage per read

In [ ]:
# ============================================================
# 32) OPTIONAL: SHOW FULL ELOVL2 COVERAGE PER READ
# ============================================================
mat_low_full, labels_low_full = build_matrix_for_plot(
    lowest_reads_show,
    region=elovl2_region,
    eligible_pos_to_col=eligible_pos_to_col,
    use_only_predicted_sites=False
)

mat_high_full, labels_high_full = build_matrix_for_plot(
    highest_reads_show,
    region=elovl2_region,
    eligible_pos_to_col=eligible_pos_to_col,
    use_only_predicted_sites=False
)

plot_read_matrix(
    mat_low_full,
    labels_low_full,
    title="ELOVL2 lowest-boundary reads, full covered eligible CpGs",
    genomic_positions=genomic_positions
)

plot_read_matrix(
    mat_high_full,
    labels_high_full,
    title="ELOVL2 highest-boundary reads, full covered eligible CpGs",
    genomic_positions=genomic_positions
)

### Single-read visualization

In [ ]:
# ============================================================
# 33) SINGLE-READ VISUALIZATION
# ============================================================
def plot_single_read(row, region, eligible_pos_to_col, eligible_pos_to_genomic_pos, use_only_predicted_sites=True):
    vec = reconstruct_read_vector_from_row(
        row=row,
        region=region,
        eligible_pos_to_col=eligible_pos_to_col,
        use_only_predicted_sites=use_only_predicted_sites
    )

    obs = np.where(~np.isnan(vec))[0]
    if len(obs) == 0:
        print("This read has no observable eligible CpGs under current setting.")
        return

    x = np.array([eligible_pos_to_genomic_pos[eligible_pos_order[i]] for i in obs])
    y = np.zeros_like(x, dtype=float)
    c = vec[obs]

    plt.figure(figsize=(10, 2.2))
    plt.scatter(x, y, c=c, cmap=plt.cm.RdBu_r, vmin=0, vmax=1, s=120, marker="s")
    plt.yticks([])
    plt.xlabel("Genomic position")
    plt.title(
        f"person_id={row['person_id']} | pred={row['pred_age']:.1f} | true={row['true_age']:.1f} | "
        f"meth={row['meth_frac']:.2f} | n={int(row['n_cpgs_used'])}"
    )
    plt.tight_layout()
    plt.show()


# example: first lowest and first highest
if len(lowest_reads_show) > 0:
    plot_single_read(
        lowest_reads_show.iloc[0],
        region=elovl2_region,
        eligible_pos_to_col=eligible_pos_to_col,
        eligible_pos_to_genomic_pos=eligible_pos_to_genomic_pos,
        use_only_predicted_sites=True
    )

if len(highest_reads_show) > 0:
    plot_single_read(
        highest_reads_show.iloc[0],
        region=elovl2_region,
        eligible_pos_to_col=eligible_pos_to_col,
        eligible_pos_to_genomic_pos=eligible_pos_to_genomic_pos,
        use_only_predicted_sites=True
    )

### A) Generic helpers to build a heatmap for all reads in a region

In [ ]:
# ============================================================
# A) GENERIC HELPERS TO BUILD A HEATMAP FOR ALL READS IN A REGION
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def get_region_object(region_name, region_objects):
    for r in region_objects:
        if r["region_name"] == region_name:
            return r
    raise ValueError(f"Region not found: {region_name}")


def get_region_sites_df(region_name, region_objects, mapped_sites):
    region = get_region_object(region_name, region_objects)
    d = mapped_sites[
        (mapped_sites["chrom"] == region["chrom"]) &
        (mapped_sites["pos"] >= region["window_start"]) &
        (mapped_sites["pos"] <= region["window_end"])
    ].copy().sort_values("pos").reset_index(drop=True)

    if len(d) == 0:
        raise ValueError(f"No eligible CpGs found in region: {region_name}")
    return d, region


def build_region_lookup_objects(region_name, region_objects, mapped_sites):
    sites_df, region = get_region_sites_df(region_name, region_objects, mapped_sites)

    eligible_pos_order = sites_df["eligible_pos"].astype(int).tolist()
    eligible_pos_to_col = {ep: i for i, ep in enumerate(eligible_pos_order)}
    eligible_pos_to_genomic_pos = dict(
        zip(sites_df["eligible_pos"].astype(int), sites_df["pos"].astype(int))
    )

    return sites_df, region, eligible_pos_order, eligible_pos_to_col, eligible_pos_to_genomic_pos


def reconstruct_read_vector_from_row(row, region, eligible_pos_to_col, use_only_predicted_sites=False):
    """
    Returns vector over region CpGs:
      1 = methylated
      0 = unmethylated
      NaN = not covered
    """
    vec = np.full(len(eligible_pos_to_col), np.nan, dtype=float)

    used_set = None
    if use_only_predicted_sites:
        used_txt = str(row["eligible_pos_used"])
        used_set = set(int(x) for x in used_txt.split(",") if x != "")

    start_global_idx1 = int(row["start_global_idx1"])
    pattern = str(row["pattern"])
    global_to_eligible = region["global_to_eligible"]

    for j, ch in enumerate(pattern):
        gidx = start_global_idx1 + j
        eligible_pos = global_to_eligible.get(gidx, None)
        if eligible_pos is None:
            continue

        if use_only_predicted_sites and (eligible_pos not in used_set):
            continue

        col = eligible_pos_to_col.get(int(eligible_pos), None)
        if col is None:
            continue

        if ch in ("C", "1"):
            vec[col] = 1.0
        elif ch in ("T", "0"):
            vec[col] = 0.0

    return vec


def build_all_read_matrix(
    reads_df,
    region_name,
    region_objects,
    mapped_sites,
    use_only_predicted_sites=False,
    sort_by="pred_age",
    max_rows_to_plot=3000,
    sample_evenly_if_large=True
):
    region_reads = reads_df[reads_df["region_name"] == region_name].copy()

    if len(region_reads) == 0:
        raise ValueError(f"No reads found for region: {region_name}")

    if sort_by is not None:
        region_reads = region_reads.sort_values(sort_by).reset_index(drop=True)

    # limit rows for plotting if too large
    if (max_rows_to_plot is not None) and (len(region_reads) > max_rows_to_plot):
        if sample_evenly_if_large:
            idx = np.linspace(0, len(region_reads) - 1, max_rows_to_plot).round().astype(int)
            region_reads = region_reads.iloc[idx].copy().reset_index(drop=True)
        else:
            region_reads = region_reads.head(max_rows_to_plot).copy().reset_index(drop=True)

    sites_df, region, eligible_pos_order, eligible_pos_to_col, eligible_pos_to_genomic_pos = \
        build_region_lookup_objects(region_name, region_objects, mapped_sites)

    mat = []
    row_meta = []

    for _, row in region_reads.iterrows():
        vec = reconstruct_read_vector_from_row(
            row=row,
            region=region,
            eligible_pos_to_col=eligible_pos_to_col,
            use_only_predicted_sites=use_only_predicted_sites
        )
        mat.append(vec)
        row_meta.append({
            "person_id": str(row["person_id"]),
            "true_age": float(row["true_age"]),
            "pred_age": float(row["pred_age"]),
            "meth_frac": float(row["meth_frac"]),
            "n_cpgs_used": int(row["n_cpgs_used"]),
            "count": int(row["count"]),
        })

    mat = np.vstack(mat)
    row_meta_df = pd.DataFrame(row_meta)

    return {
        "matrix": mat,
        "row_meta_df": row_meta_df,
        "sites_df": sites_df,
        "region": region,
        "eligible_pos_order": eligible_pos_order,
        "eligible_pos_to_genomic_pos": eligible_pos_to_genomic_pos,
    }

### B) Plot raw read x CpG heatmap

In [ ]:
# ============================================================
# B) PLOT RAW READ x CpG HEATMAP
# ============================================================
def plot_read_heatmap(
    heatmap_obj,
    title,
    low_cut=None,
    high_cut=None,
    show_yaxis=False
):
    mat = heatmap_obj["matrix"]
    row_meta_df = heatmap_obj["row_meta_df"]
    eligible_pos_order = heatmap_obj["eligible_pos_order"]
    eligible_pos_to_genomic_pos = heatmap_obj["eligible_pos_to_genomic_pos"]

    genomic_positions = [eligible_pos_to_genomic_pos[ep] for ep in eligible_pos_order]

    cmap = plt.cm.RdBu_r.copy()
    cmap.set_bad("lightgray")

    fig_h = max(5, min(14, mat.shape[0] * 0.03))
    fig_w = max(10, mat.shape[1] * 0.35)

    plt.figure(figsize=(fig_w, fig_h))
    im = plt.imshow(mat, aspect="auto", interpolation="nearest", cmap=cmap, vmin=0, vmax=1)

    if show_yaxis:
        ylabels = [
            f"{r.person_id} | p={r.pred_age:.1f} | t={r.true_age:.1f}"
            for r in row_meta_df.itertuples(index=False)
        ]
        plt.yticks(np.arange(len(ylabels)), ylabels, fontsize=6)
    else:
        plt.yticks([])

    xticks = np.arange(len(genomic_positions))
    xticklabels = [str(x) for x in genomic_positions]

    if len(xticks) > 12:
        keep = np.linspace(0, len(xticks) - 1, 12).round().astype(int)
        plt.xticks(keep, [xticklabels[i] for i in keep], rotation=90, fontsize=8)
    else:
        plt.xticks(xticks, xticklabels, rotation=90, fontsize=8)

    # boundary lines if requested
    if low_cut is not None:
        low_rows = np.where(row_meta_df["pred_age"].to_numpy() <= low_cut)[0]
        if len(low_rows) > 0:
            plt.axhline(low_rows.max() + 0.5, color="black", linestyle="--", linewidth=1.5)

    if high_cut is not None:
        high_rows = np.where(row_meta_df["pred_age"].to_numpy() >= high_cut)[0]
        if len(high_rows) > 0:
            plt.axhline(high_rows.min() - 0.5, color="black", linestyle="--", linewidth=1.5)

    plt.xlabel("Eligible CpG genomic position")
    plt.ylabel("Reads sorted by predicted age")
    plt.title(title)
    plt.text(mat.shape[1] + 0.2, low_rows.max() / 2, "low boundary", va="center")
    plt.text(mat.shape[1] + 0.2, (low_rows.max() + high_rows.min()) / 2, "middle", va="center")
    plt.text(mat.shape[1] + 0.2, (high_rows.min() + mat.shape[0]) / 2, "high boundary", va="center")
    cbar = plt.colorbar(im)
    cbar.set_label("Methylation state (0 = unmethylated/T, 1 = methylated/C)")

    plt.tight_layout()
    plt.show()

### C) Plot binned summary heatmap

In [ ]:
# ============================================================
# C) PLOT BINNED SUMMARY HEATMAP
# ============================================================
def plot_binned_methylation_heatmap(
    heatmap_obj,
    title,
    n_bins=12,
    weighted_by_count=False
):
    mat = heatmap_obj["matrix"]
    row_meta_df = heatmap_obj["row_meta_df"]
    eligible_pos_order = heatmap_obj["eligible_pos_order"]
    eligible_pos_to_genomic_pos = heatmap_obj["eligible_pos_to_genomic_pos"]

    genomic_positions = [eligible_pos_to_genomic_pos[ep] for ep in eligible_pos_order]

    df = row_meta_df.copy()
    df["bin"] = pd.qcut(df["pred_age"], q=min(n_bins, len(df)), duplicates="drop")

    bin_labels = []
    bin_means = []

    for b, idx in df.groupby("bin").groups.items():
        idx = np.array(list(idx), dtype=int)
        sub = mat[idx, :]

        if weighted_by_count:
            w = row_meta_df.iloc[idx]["count"].to_numpy(dtype=float)
            mean_vec = np.array([
                np.average(col[~np.isnan(col)], weights=w[~np.isnan(col)]) if np.sum(~np.isnan(col)) > 0 else np.nan
                for col in sub.T
            ])
        else:
            mean_vec = np.nanmean(sub, axis=0)

        bin_labels.append(f"{b.left:.1f} to {b.right:.1f}")
        bin_means.append(mean_vec)

    bin_means = np.vstack(bin_means)

    cmap = plt.cm.RdBu_r.copy()
    cmap.set_bad("lightgray")

    fig_w = max(10, len(genomic_positions) * 0.35)
    fig_h = max(4, len(bin_labels) * 0.45)

    plt.figure(figsize=(fig_w, fig_h))
    im = plt.imshow(bin_means, aspect="auto", interpolation="nearest", cmap=cmap, vmin=0, vmax=1)

    plt.yticks(np.arange(len(bin_labels)), bin_labels, fontsize=8)

    xticks = np.arange(len(genomic_positions))
    xticklabels = [str(x) for x in genomic_positions]

    if len(xticks) > 12:
        keep = np.linspace(0, len(xticks) - 1, 12).round().astype(int)
        plt.xticks(keep, [xticklabels[i] for i in keep], rotation=90, fontsize=8)
    else:
        plt.xticks(xticks, xticklabels, rotation=90, fontsize=8)

    plt.xlabel("Eligible CpG genomic position")
    plt.ylabel("Predicted-age bins")
    plt.title(title)

    cbar = plt.colorbar(im)
    cbar.set_label("Mean methylation")

    plt.tight_layout()
    plt.show()

### D) ELOVL2 all-read heatmaps

In [ ]:
# ============================================================
# D) ELOVL2 ALL-READ HEATMAPS
# ============================================================
ELOVL2_REGION_NAME = "ELOVL2_pm5kb"

elovl2_reads_all = reads_df[reads_df["region_name"] == ELOVL2_REGION_NAME].copy()
print("Total ELOVL2 read rows:", len(elovl2_reads_all))

low_cut = elovl2_reads_all["pred_age"].quantile(BOUNDARY_Q_LOW)
high_cut = elovl2_reads_all["pred_age"].quantile(BOUNDARY_Q_HIGH)

# 1. Raw read x CpG heatmap, using all covered eligible CpGs
elovl2_heatmap_full = build_all_read_matrix(
    reads_df=reads_df,
    region_name=ELOVL2_REGION_NAME,
    region_objects=region_objects,
    mapped_sites=mapped_sites,
    use_only_predicted_sites=False,
    sort_by="pred_age",
    max_rows_to_plot=3000
)

plot_read_heatmap(
    elovl2_heatmap_full,
    title="ELOVL2 all-read heatmap, full covered eligible CpGs",
    low_cut=low_cut,
    high_cut=high_cut,
    show_yaxis=False
)

# 2. Same heatmap, but only the CpGs actually used in prediction
elovl2_heatmap_used = build_all_read_matrix(
    reads_df=reads_df,
    region_name=ELOVL2_REGION_NAME,
    region_objects=region_objects,
    mapped_sites=mapped_sites,
    use_only_predicted_sites=True,
    sort_by="pred_age",
    max_rows_to_plot=3000
)

plot_read_heatmap(
    elovl2_heatmap_used,
    title="ELOVL2 all-read heatmap, predicted CpGs only",
    low_cut=low_cut,
    high_cut=high_cut,
    show_yaxis=False
)

# 3. Binned summary heatmap
plot_binned_methylation_heatmap(
    elovl2_heatmap_full,
    title="ELOVL2 binned summary heatmap by predicted age",
    n_bins=12,
    weighted_by_count=False
)

### Clean binary read heatmap (no colorbar, dashed separator)

In [ ]:
# ============================================================
# CLEAN BINARY READ HEATMAP
#  - no colorbar
#  - dashed separator lines
#  - low / middle / high labels without ugly extra box
#  - explicit 3-state legend
#  - optional silent known-site marker
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch


def plot_binary_read_heatmap_clean(
    heatmap_obj,
    title,
    low_cut=None,
    high_cut=None,
    show_yaxis=False,
    known_site_pos=11044644,   # set to None to remove marker
    label_side="right",        # "right" or "left"
):
    mat = heatmap_obj["matrix"]
    row_meta_df = heatmap_obj["row_meta_df"].copy()
    eligible_pos_order = heatmap_obj["eligible_pos_order"]
    eligible_pos_to_genomic_pos = heatmap_obj["eligible_pos_to_genomic_pos"]

    genomic_positions = np.array(
        [eligible_pos_to_genomic_pos[ep] for ep in eligible_pos_order],
        dtype=int
    )

    n_rows, n_cols = mat.shape

    # 0 = unmethylated, 1 = methylated, NaN = missing/not covered
    cmap = ListedColormap(["#08306b", "#b2182b"])
    cmap.set_bad("white")

    fig_h = max(5, min(14, n_rows * 0.03))
    fig_w = max(10, n_cols * 0.45)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    ax.imshow(
        mat,
        aspect="auto",
        interpolation="nearest",
        cmap=cmap,
        vmin=0,
        vmax=1,
        origin="upper"
    )

    # ------------------------------------------------
    # y-axis
    # ------------------------------------------------
    if show_yaxis:
        ylabels = [
            f"{r.person_id} | p={r.pred_age:.1f} | t={r.true_age:.1f}"
            for r in row_meta_df.itertuples(index=False)
        ]
        ax.set_yticks(np.arange(len(ylabels)))
        ax.set_yticklabels(ylabels, fontsize=6)
    else:
        ax.set_yticks([])

    # ------------------------------------------------
    # x-axis
    # ------------------------------------------------
    xticks = np.arange(len(genomic_positions))
    xticklabels = [str(x) for x in genomic_positions]

    if len(xticks) > 12:
        keep = np.linspace(0, len(xticks) - 1, 12).round().astype(int)
        ax.set_xticks(keep)
        ax.set_xticklabels([xticklabels[i] for i in keep], rotation=90, fontsize=8)
    else:
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, rotation=90, fontsize=8)

    # ------------------------------------------------
    # boundary separator lines
    # ------------------------------------------------
    pred = row_meta_df["pred_age"].to_numpy(dtype=float)

    low_end = None
    high_start = None

    if low_cut is not None:
        low_rows = np.where(pred <= low_cut)[0]
        if len(low_rows) > 0:
            low_end = low_rows.max() + 0.5
            ax.axhline(
                low_end,
                color="black",
                linestyle="--",
                linewidth=1.2
            )

    if high_cut is not None:
        high_rows = np.where(pred >= high_cut)[0]
        if len(high_rows) > 0:
            high_start = high_rows.min() - 0.5
            ax.axhline(
                high_start,
                color="black",
                linestyle="--",
                linewidth=1.2
            )

    # ------------------------------------------------
    # low / middle / high labels
    # Use blended transform:
    #   x in axes coordinates
    #   y in data coordinates
    # so labels do not require changing xlim
    # ------------------------------------------------
    if low_end is not None and high_start is not None:
        low_center = low_end / 2.0
        middle_center = (low_end + high_start) / 2.0
        high_center = (high_start + (n_rows - 1)) / 2.0

        trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)

        if label_side == "right":
            label_x = 1.01
            ha = "left"
        else:
            label_x = -0.01
            ha = "right"

        ax.text(
            label_x, low_center, "low",
            transform=trans, va="center", ha=ha, fontsize=10, clip_on=False
        )
        ax.text(
            label_x, middle_center, "mid",
            transform=trans, va="center", ha=ha, fontsize=10, clip_on=False
        )
        ax.text(
            label_x, high_center, "high",
            transform=trans, va="center", ha=ha, fontsize=10, clip_on=False
        )

    # ------------------------------------------------
    # optional silent known-site marker
    # ------------------------------------------------
    if known_site_pos is not None and len(genomic_positions) > 0:
        nearest_idx = int(np.argmin(np.abs(genomic_positions - known_site_pos)))
        ax.axvline(nearest_idx, color="black", linestyle=":", linewidth=1.5)

    # ------------------------------------------------
    # labels and legend
    # ------------------------------------------------
    ax.set_xlabel("Eligible CpG genomic position")
    ax.set_ylabel("Reads sorted by predicted age")
    ax.set_title(title)

    legend_handles = [
        Patch(facecolor="#08306b", edgecolor="black", label="Unmethylated (T / 0)"),
        Patch(facecolor="#b2182b", edgecolor="black", label="Methylated (C / 1)"),
        Patch(facecolor="white", edgecolor="black", label="Not covered / missing"),
    ]

    ax.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=3,
        frameon=False,
        fontsize=10
    )

    # Leave a little room for labels on the chosen side
    if label_side == "right":
        fig.subplots_adjust(right=0.88, bottom=0.16)
    else:
        fig.subplots_adjust(left=0.12, bottom=0.16)

    plt.show()